In [1]:
import sys
# Add the desired directory to sys.path
sys.path.append('build/')

import randSVD
from sklearn.utils.extmath import randomized_svd

seed = 7050
tol = 1e-3

## Test matrix

In [2]:
import numpy as np

class TestMatrix:
    def __init__(self, rows, cols, sing_val_decay):
        self.rows = rows
        self.cols = cols
        self.sing_val_decay = sing_val_decay
        
        # Calculate rank
        self.rank = min(rows, cols)
        
        # Generate random U and V matrices
        self.U = np.random.rand(rows, self.rank)
        self.V = np.random.rand(cols, self.rank)
        
        # Orthogonalize U and V using QR decomposition
        self.U, _ = np.linalg.qr(self.U)
        self.V, _ = np.linalg.qr(self.V)
        
        # Set the singular values based on the decay option
        if self.sing_val_decay == "fast":
            self.sing_vals = np.power(0.95, np.arange(self.rank))
        else:
            self.sing_vals = np.log(2 + np.arange(self.rank))

        self.A = (self.U @ np.diag(self.sing_vals) @ self.V.T).astype(np.float64)
        
    def matrixU(self):
        return self.U
    
    def singularValues(self):
        return self.sing_vals
    
    def matrixV(self):
        return self.V
    
    def matrixA(self):
        return self.A
    

## Error metrics

In [3]:
def compute_errors(test_matrix,U,sing_vals,V):
    errors = {}

    rank = sing_vals.size

    errors["rec_error"] = np.linalg.norm(test_matrix.matrixA() - U@np.diag(sing_vals)@V.T, 'fro')
    errors["sing_val_error"] = np.linalg.norm(test_matrix.singularValues()[:rank] - sing_vals)

    errors["l_sing_vect_error"] = min(max(np.linalg.norm(test_matrix.matrixU()[:,:rank]-U,axis=0)),max(np.linalg.norm(test_matrix.matrixU()[:,:rank]+U,axis=0)))
    errors["r_sing_vect_error"] = min(max(np.linalg.norm(test_matrix.matrixV()[:,:rank]-V,axis=0)),max(np.linalg.norm(test_matrix.matrixV()[:,:rank]+V,axis=0)))

    return errors

## Computations

In [4]:
import pandas as pd
# Assuming randSVD and TestMatrix are already defined somewhere

test_results = pd.DataFrame(columns=['svd', 'size','replica','rec_error', 'sing_val_error', 'l_sing_vect_error', 'r_sing_vect_error'])
tested_sizes = [1000,2000,4000,8000]

rank = 10 #to compare with rbki
n_iter = 5
tol = 1e-10
seed = 7050

n_replicas = 10

rsi = randSVD.RSI(seed, tol)
rbki = randSVD.RBKI(seed, tol)

for test_size in tested_sizes:
    print("Size: ",test_size)
    test_matrix = TestMatrix(test_size, test_size, "fast")
    
    for i in range(1,n_replicas+1):

        # RSI computation
        rsi.compute(test_matrix.matrixA(), rank, n_iter)
        test_results = pd.concat([test_results, pd.DataFrame([["rsi", test_size,i]], columns=['svd','size','replica'])], ignore_index=True)
        errors_rsi = compute_errors(test_matrix, rsi.matrixU(), rsi.singularValues(), rsi.matrixV())
        test_results.loc[(test_results['svd'] == 'rsi') & (test_results['size'] == test_size) & (test_results['replica'] == i), ['rec_error', 'sing_val_error', 'l_sing_vect_error', 'r_sing_vect_error']] = list(errors_rsi.values())
        
        # RBKI computation
        rbki.compute(test_matrix.matrixA(), rank, n_iter)
        test_results = pd.concat([test_results, pd.DataFrame([["rbki", test_size,i]], columns=['svd','size','replica'])], ignore_index=True)
        errors_rbki = compute_errors(test_matrix, rbki.matrixU(), rbki.singularValues(), rbki.matrixV())
        test_results.loc[(test_results['svd'] == 'rbki') & (test_results['size'] == test_size) & (test_results['replica'] == i), ['rec_error', 'sing_val_error', 'l_sing_vect_error', 'r_sing_vect_error']] = list(errors_rbki.values())
        
        # Scikit-learn computation (randomized SVD)
        U, s, Vh = randomized_svd(test_matrix.matrixA(),
                                n_components=rank,
                                n_oversamples=rank,
                                n_iter=n_iter,
                                random_state=0)
        test_results = pd.concat([test_results, pd.DataFrame([["scikit-learn", test_size,i]], columns=['svd','size','replica'])], ignore_index=True)
        errors_sklearn = compute_errors(test_matrix, U, s, Vh.T)
        test_results.loc[(test_results['svd'] == 'scikit-learn') & (test_results['size'] == test_size) & (test_results['replica'] == i), ['rec_error', 'sing_val_error', 'l_sing_vect_error', 'r_sing_vect_error']] = list(errors_sklearn.values())

Size:  1000
Size:  2000
Size:  4000
Size:  8000


In [5]:
print(test_results)
test_results.to_csv('results/randSVD_comparison.csv', index=False)

              svd  size replica rec_error sing_val_error l_sing_vect_error  \
0             rsi  1000       1  1.917493            0.0               2.0   
1            rbki  1000       1  1.917493            0.0               2.0   
2    scikit-learn  1000       1  1.917493       0.000001               2.0   
3             rsi  1000       2  1.917493            0.0               2.0   
4            rbki  1000       2  1.917493            0.0               2.0   
..            ...   ...     ...       ...            ...               ...   
115          rbki  8000       9  1.917493            0.0               2.0   
116  scikit-learn  8000       9  1.917493       0.000001               2.0   
117           rsi  8000      10  1.917493       0.000001               2.0   
118          rbki  8000      10  1.917493            0.0               2.0   
119  scikit-learn  8000      10  1.917493       0.000001               2.0   

    r_sing_vect_error  
0                 2.0  
1              